## ⚠️ Notebook status (updated dataset)

These notebooks were originally created as experiments (02–14). Their historical runs used **deprecated / non-integral data**.

As of **May 24, 2026**, the integral dataset is:
`dataset/raw/handover_dataset.csv`

Important dataset property:
- `optimal_cell_idx_in_k` is **constant 0** in the integral dataset because neighbor lists are score-sorted with the optimal cell at index 0.
- Any pipeline that trains directly on `optimal_cell_idx_in_k` without **neighbor-axis shuffling** will learn order leakage and produce misleading accuracy.

Recommended usage now:
- Treat notebooks 02–14 as **robustness / stability / scalability** harnesses.
- For a leakage-safe pointer target, reuse `src/preprocess.py` (`dataset/processed/*.npz`) or `src/production/temporal_deepset_data.py`.

See production docs:
- `docs/production_constraints.md`
- `docs/robustness_scalability_plan.md`
- `docs/explainability_and_finetuning.md`


In [24]:
# --- Multi-horizon window controls (NEW DATASET) ---
# Each row in the dataset corresponds to one measurement interval.
MEASUREMENT_INTERVAL_MS = 50

# History window length (timesteps). Example: 25 → 1.25 s history @ 50 ms.
WIN_T = 25

# Multi-horizon label generation. Example: 5 → predict up to 250 ms ahead.
FUTURE_H = 5

# Optional lead time before horizon starts (in timesteps).
LEAD_L = 0

# For notebooks that are NOT multi-output, choose which horizon to train on (1..FUTURE_H).
TARGET_H_IDX = 1

# Label source:f
# - "optimal": train against oracle best cell (optimal_cell_id)
# - "target" : train against executed target (target_cell_id)
LABEL_MODE = "optimal"

# Feature toggles for cache builder:
# - include_scores=True adds nb_scores to per-cell features
# - include_global=True adds [speed, cos(dir), sin(dir), cell_load] to each cell feature vector
INCLUDE_SCORES = True
INCLUDE_GLOBAL = False

# Set True to force rebuilding dataset/mh_cache for new window/horizon settings
FORCE_REBUILD = True

# Use the leakage-safe multi-horizon cache loader.
USE_MH_CACHE = True

print(
    f"[window] dt={MEASUREMENT_INTERVAL_MS}ms  "
    f"T={WIN_T} ({WIN_T*MEASUREMENT_INTERVAL_MS}ms)  "
    f"H={FUTURE_H} ({FUTURE_H*MEASUREMENT_INTERVAL_MS}ms)  "
    f"lead={LEAD_L}  target_h={TARGET_H_IDX}"
)


[window] dt=50ms  T=25 (1250ms)  H=5 (250ms)  lead=0  target_h=1


In [25]:
# ─── SECTION 4: UNIFIED DATA PIPELINE (Multi-Horizon & Strictly Balanced) ───
# Inspired by 01_temporal_deepset.ipynb, but generalized for Multi-Horizon
# Provides X, M, y (multi-horizon), r (regression), and y_bin (binary HO).

SEED = 42
from pathlib import Path
import logging
try: _r = _ROOT
except NameError: _r = Path("../../").resolve()
try: log.info
except NameError: log = logging.getLogger("dummy"); log.setLevel(logging.INFO)
import re
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

def parse_nb_array(s, max_len=10, fill=0.0):
    if pd.isna(s): return [fill] * max_len
    cleaned = re.sub(r'[\[\]]', '', str(s)).strip()
    parts = re.split(r'[,;]', cleaned)
    vals = []
    for p in parts[:max_len]:
        p = p.strip()
        if p == '' or p.lower() in ('nan', 'none'): vals.append(fill)
        else:
            try: vals.append(float(p))
            except: vals.append(fill)
    vals += [fill] * (max_len - len(vals))
    return vals

def parse_nb_ids(s, max_len=10):
    if pd.isna(s): return [0] * max_len
    cleaned = re.sub(r'[\[\]]', '', str(s)).strip()
    parts = re.split(r'[,;]', cleaned)
    ids = []
    for p in parts[:max_len]:
        p = p.strip()
        try: ids.append(int(float(p)))
        except: ids.append(0)
    ids += [0] * (max_len - len(ids))
    return ids

def load_and_create_mh_datasets(root_dir, win_t=25, future_h=5, lead_l=0, k_cells=10):
    raw_path = root_dir / "dataset" / "raw" / "handover_dataset.csv"
    log.info(f"Loading raw data from {raw_path}...")
    df = pd.read_csv(raw_path, low_memory=False)
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed")
    
    df["nb_ids"]   = df["nb_cell_ids"].apply(parse_nb_ids)
    df["nb_rsrps"] = df["nb_rsrps"].apply(parse_nb_array)
    df["nb_sinrs"] = df["nb_sinrs"].apply(parse_nb_array)
    df["nb_loads"] = df["nb_loads"].apply(parse_nb_array)
    
    df.sort_values(["ue_id", "timestamp"], inplace=True)
    
    all_X, all_M, all_y, all_r, groups = [], [], [], [], []
    rng_shuf = np.random.default_rng(SEED)
    
    log.info(f"Building MH sequences (T={win_t}, H={future_h}, L={lead_l}) with mandatory shuffling...")
    for ue_id, grp in df.groupby("ue_id", sort=False):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        n_rows = len(grp)
        if n_rows < win_t + lead_l + future_h: continue
        
        cell_feat = np.zeros((n_rows, k_cells, 4), dtype=np.float32)
        for i in range(n_rows):
            rs, sn, ld = grp.at[i, "nb_rsrps"], grp.at[i, "nb_sinrs"], grp.at[i, "nb_loads"]
            # 4th feature is placeholder for score if needed, or 0
            for k in range(k_cells):
                cell_feat[i, k, 0] = rs[k]
                cell_feat[i, k, 1] = sn[k]
                cell_feat[i, k, 2] = ld[k]
                cell_feat[i, k, 3] = 0.0 # score placeholder
                
        opt_ids = grp["optimal_cell_id"].values
        ue_nb_ids = grp["nb_ids"].values
        ue_srv_ids = grp["serving_cell_id"].values
        opt_rsrp = grp["optimal_cell_rsrp"].values
        
        for t in range(win_t, n_rows - (lead_l + future_h) + 1):
            X_w = cell_feat[t-win_t : t] 
            p = rng_shuf.permutation(k_cells)
            X_w_shuf = X_w[:, p, :].transpose(1, 0, 2)
            M_w = (X_w_shuf[:, -1, 0] != 0.0).astype(np.float32)
            
            yh = np.zeros((future_h,), dtype=np.int32)
            rh = np.zeros((future_h,), dtype=np.float32)
            
            current_nb_ids = ue_nb_ids[t-1]
            
            for step in range(future_h):
                f_idx = t + lead_l + step
                chosen_id = int(opt_ids[f_idx])
                rh[step] = float(opt_rsrp[f_idx])
                try:
                    orig = current_nb_ids.index(chosen_id)
                    yh[step] = int(np.where(p == orig)[0][0])
                except ValueError:
                    srv = int(ue_srv_ids[f_idx])
                    try:
                        orig = current_nb_ids.index(srv)
                        yh[step] = int(np.where(p == orig)[0][0])
                    except ValueError:
                        yh[step] = 0
            
            all_X.append(X_w_shuf)
            all_M.append(M_w)
            all_y.append(yh)
            all_r.append(rh)
            groups.append(ue_id)

    X = np.array(all_X)
    M = np.array(all_M)
    y = np.array(all_y)
    r = np.array(all_r, dtype=np.float32)
    groups = np.array(groups)
    
    # Split
    ue_list = np.unique(groups)
    np.random.default_rng(SEED).shuffle(ue_list)
    n_te, n_va = int(len(ue_list)*0.15), int(len(ue_list)*0.15)
    ue_te, ue_va = set(ue_list[:n_te]), set(ue_list[n_te:n_te+n_va])
    
    idx_tr = np.where([u not in ue_te and u not in ue_va for u in groups])[0]
    idx_va = np.where([u in ue_va for u in groups])[0]
    idx_te = np.where([u in ue_te for u in groups])[0]
    
    # Scale Features
    scaler_x = StandardScaler()
    scaler_x.fit(X[idx_tr][M[idx_tr]==1].reshape(-1, 4))
    X_n = X.copy()
    for i in range(len(X_n)):
        vk = np.where(M[i] == 1.0)[0]
        if len(vk) > 0:
            X_n[i, vk] = scaler_x.transform(X[i, vk].reshape(-1, 4)).reshape(-1, win_t, 4)
            
    # Scale Regression
    scaler_r = StandardScaler()
    scaler_r.fit(r[idx_tr].reshape(-1, 1))
    r_n = scaler_r.transform(r.reshape(-1, 1)).reshape(-1, future_h)
    
    return {
        "train": {"X": X_n[idx_tr], "M": M[idx_tr], "y": y[idx_tr], "r": r_n[idx_tr]},
        "val":   {"X": X_n[idx_va], "M": M[idx_va], "y": y[idx_va], "r": r_n[idx_va]},
        "test":  {"X": X_n[idx_te], "M": M[idx_te], "y": y[idx_te], "r": r_n[idx_te]},
        "scalers": {"x": scaler_x, "r": scaler_r}
    }

# Execute unified pipeline
try: _wt = WIN_T
except: _wt = HP.get("OBS_STEPS", 25)
try: _fh = FUTURE_H
except: _fh = 5
try: _ll = LEAD_L
except: _ll = 0

_data = load_and_create_mh_datasets(_r, win_t=_wt, future_h=_fh, lead_l=_ll, k_cells=10)

# Map to canonical variables
X_tr, M_tr, y_tr_mh, r_tr = _data["train"]["X"], _data["train"]["M"], _data["train"]["y"], _data["train"]["r"]
X_va, M_va, y_va_mh, r_va = _data["val"]["X"], _data["val"]["M"], _data["val"]["y"], _data["val"]["r"]
X_te, M_te, y_te_mh, r_te = _data["test"]["X"], _data["test"]["M"], _data["test"]["y"], _data["test"]["r"]
scaler = _data["scalers"]["x"]

# Multi-Horizon: preserve all H steps
y_tr = y_tr_mh.astype("int32")
y_va = y_va_mh.astype("int32")
y_te = y_te_mh.astype("int32")

# If notebook expects binary HO labels
y_bin_tr = (y_tr > 0).astype(np.float32)
y_bin_va = (y_va > 0).astype(np.float32)
y_bin_te = (y_te > 0).astype(np.float32)

# Ensure N_FEATS matches our 4-feature output (which might have 3 valid ones)
try:
    if "N_FEATS" in HP:
        HP["N_FEATS"] = 4
except NameError:
    pass

log.info("Unified MH Datasets Ready! X_tr: %s, y_tr: %s", X_tr.shape, y_tr.shape)



11:08:13 │ INFO     │ Loading raw data from /home/wassimmchichi/Downloads/Handover_projects/dataset/raw/handover_dataset.csv...
11:08:18 │ INFO     │ Building MH sequences (T=25, H=5, L=0) with mandatory shuffling...
11:08:42 │ INFO     │ Unified MH Datasets Ready! X_tr: (57120, 10, 25, 4), y_tr: (57120, 5)


# 05· Strategic DeepSet — Breaking the 55% Top-1 Plateau

## Three-axis diagnosis of the current failure

```
SYMPTOM: Top-3 = 86%, Top-1 = 55%
→ The correct cell is in the top-3 almost always.
→ The model lacks "Strategic Logic" to break the tie.

AXIS 1 — Missing context          AXIS 2 — Noisy supervision         AXIS 3 — Class imbalance
───────────────────────────       ─────────────────────────────       ────────────────────────
Model sees RF quality only.       Learning from failed HOs             Cell 0 recall = 83%.
Ignores UE trajectory intent,     and ping-pong events injects         Cells 2-7 recall < 30%.
network congestion, and the        contradictory gradient signal        Focal Loss needed.
simulator's policy logic.          that defeats generalisation.
```

## What this notebook implements

| Component | Previous | This notebook |
|---|---|---|
| Cell features | `(B,10,25,4)` RF only | `(B,10,25,4)` + Global State `(B,25,9)` |
| Global State | None | `[speed, dir_cos, dir_sin, cell_load, ho_class×5]` |
| Training data | All 47,920 rows | Sanitized: `success=1` OR `no_handover`, `ping_pong=0` |
| Loss | CCE | Focal + Huber(RSRP) + sample weights in `tf.data` |
| Architecture | `z = GlobalAvgPool` | `z_aug = [GlobalAvgPool ‖ GlobalState_LSTM]` |
| Rho head | Dense 64 | Dense 256 → 128 (handles 137-dim concat) |

## Feature engineering rationale

```
Global State vector (per timestep):
  [0] speed     → trajectory intent: fast UE = imminent coverage change
  [1] dir_cos   → sin/cos encoding of heading (no 0°/360° discontinuity)
  [2] dir_sin
  [3] cell_load → serving cell congestion (high load = push UE away)
  [4] ho_class_0    → no_handover   (one-hot of simulator policy decision)
  [5] ho_class_1    → coverage HO
  [6] ho_class_3    → inter-RAT HO
  [7] ho_class_5    → emergency HO
  [8] ho_class_6    → RLF HO

Why one-hot `handover_class`?
  It encodes the simulator's INTENT — coverage-driven vs load-balancing.
  A load-balancing HO selects a different cell than a coverage HO even
  for the same RSRP snapshot. This is the "Strategic Logic" the model
  was missing.
```

## Section 1 · Environment, Paths, GPU, MLflow

In [ ]:
import os, sys, warnings, json, pickle, logging, datetime, gc, re, time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing  import Tuple, List, Dict

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from sklearn.preprocessing      import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics            import (classification_report, confusion_matrix,
                                        top_k_accuracy_score, mean_absolute_error)
sns.set_theme(style="whitegrid", font_scale=1.05)

_ROOT = Path("../../").resolve()

PATHS = dict(
    raw_csv  = _ROOT / "dataset" / "raw"            / "handover_dataset.csv",
    cache    = _ROOT / "dataset" / "strategic_cache",
    models   = _ROOT / "models",
    metrics  = _ROOT / "metrics"/"strategic_deepset",
    tb_logs  = _ROOT / "tb_logs" / "strategic_deepset",
    mlruns   = _ROOT / "mlflow"  / "mlruns",
)
for p in (PATHS["cache"], PATHS["models"],
          PATHS["metrics"], PATHS["tb_logs"], PATHS["mlruns"]):
    os.makedirs(str(p), exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s │ %(levelname)-8s │ %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(str(PATHS["metrics"]/"strategic_training.log"), mode="w")]
)
log = logging.getLogger("strategic")
log.info("Root: %s", _ROOT)

gpus = tf.config.list_physical_devices("GPU")
log.info("GPUs: %d", len(gpus))
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
if not gpus:
    log.warning("No GPU detected.")

policy = mixed_precision.Policy("mixed_float16")
mixed_precision.set_global_policy(policy)
log.info("Mixed precision: compute=%s  vars=%s",
         policy.compute_dtype, policy.variable_dtype)

_MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
try:
    import mlflow, mlflow.tensorflow, requests
    MLFLOW_OK = requests.get(f"{_MLFLOW_URI}/health", timeout=3).status_code == 200
    if MLFLOW_OK:
        mlflow.set_tracking_uri(_MLFLOW_URI)
        mlflow.set_experiment("andover_Strategic_DeepSet")
        log.info("MLflow → %s", _MLFLOW_URI)
    else:
        log.warning("MLflow unreachable — CSV-only logging.")
except Exception:
    MLFLOW_OK = False

SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)
log.info("TF %s | NumPy %s", tf.__version__, np.__version__)

11:08:42 │ INFO     │ Root: /home/wassimmchichi/Downloads/Handover_projects
11:08:42 │ INFO     │ GPUs: 1
11:08:42 │ INFO     │ Mixed precision: compute=float16  vars=float32


2026/05/30 11:08:42 INFO mlflow.tracking.fluent: Experiment with name 'handover_Strategic_DeepSet' does not exist. Creating a new experiment.


11:08:42 │ INFO     │ MLflow → http://127.0.0.1:5000
11:08:42 │ INFO     │ TF 2.15.1 | NumPy 1.26.4


## Section 2 · Hyperparameters

In [27]:
HP = dict(
    # ── Data ──────────────────────────────────────────────────────────────────
    MAX_CELLS   = 10,
    OBS_STEPS   = 25,     # 5s @ 5 Hz
    F_CELL      = 3,      # per-cell: [nb_rsrp, nb_sinr, nb_load]  (nb_score removed — leakage)
    G_DIM       = 9,      # global state: [speed, dir_cos, dir_sin, cell_load, ho_class×5]

    # ── Loss weights ──────────────────────────────────────────────────────────
    LAMBDA_CLS  = 1.0,
    LAMBDA_REG  = 0.5,
    FOCAL_GAMMA = 2.0,
    FOCAL_ALPHA = 0.25,
    HUBER_DELTA = 1.0,

    # ── Architecture ──────────────────────────────────────────────────────────
    LSTM_CELL   = 64,     # temporal encoder per cell
    LSTM_GLOB   = 32,     # global state encoder
    PHI_DIM     = 64,     # Φ projection
    RHO_DIMS    = [256, 128],  # rho head — enlarged for 137-dim input
    DROPOUT     = 0.20,

    # ── Training ──────────────────────────────────────────────────────────────
    BATCH_SIZE  = 128,
    EPOCHS      = 60,
    LR_INIT     = 5e-4,
    LR_WARMUP_EP= 4,
    LR_DECAY_EP = 20,
)

ALL_LABELS = list(range(HP["MAX_CELLS"]))
CKPT_PATH  = str(PATHS["models"] / "best_strategic_deepset.keras")
FINAL_PATH = str(PATHS["models"] / "strategic_deepset_final.keras")

log.info("HP: F_CELL=%d  G_DIM=%d  RHO=%s",
         HP["F_CELL"], HP["G_DIM"], HP["RHO_DIMS"])

# --- Override window controls (injected) ---
try:
    HP["OBS_STEPS"] = int(WIN_T)
    if "PRED_STEPS" in HP: HP["PRED_STEPS"] = int(FUTURE_H)
    if "TGT_STEPS"  in HP: HP["TGT_STEPS"]  = int(FUTURE_H)
    if "LAT_STEPS"  in HP: HP["LAT_STEPS"]  = int(LEAD_L)
except Exception as _e:
    print('HP override skipped:', _e)


11:08:42 │ INFO     │ HP: F_CELL=3  G_DIM=9  RHO=[256, 128]


## Section 3 · Data Sanitization Pipeline — "Perfect Label" Hypothesis

### Why filtering matters

The model currently trains on ALL 47,920 rows, including:
- Failed handovers (`handover_success=0, handover_class>0`) → 857 samples  
  These are cases where the simulator tried to HO but the radio link failed.
  The label says "go to Cell X" but Cell X was actually unreachable.
  
- Ping-pong events (`ping_pong_flag=1`) → 857 samples  
  The UE oscillated between two cells within the TTT window.
  The label is ambiguous — neither cell was clearly better.

**Training on these generates contradictory gradients**: the model learns
to predict cells that are sometimes physically reachable and sometimes not,
creating a noisy loss landscape that prevents convergence above 55%.

### Sanitization rule
```python
keep = (handover_success == 1)  OR  (handover_class == 0)   # success + no-HO
     AND (ping_pong_flag == 0)                                # no oscillation
```
This retains 94.9% of data while removing all ambiguous supervision.

In [28]:
# ─── Section 3 · Vectorised preprocessing pipeline ───────────────────────────

_BRACKET = re.compile(r'[\[\]]')

def parse_list_col(series: pd.Series, k: int = HP["MAX_CELLS"]) -> np.ndarray:
    """Parse entire column of '[v0;v1;…]' strings → (N, k) float32 array."""
    N = len(series); out = np.full((N, k), np.nan, dtype=np.float32)
    for i, raw in enumerate(series):
        s = _BRACKET.sub("", str(raw)).strip()
        for j, p in enumerate(s.split(";")[:k]):
            try: out[i, j] = float(p.strip())
            except ValueError: pass
    return out


def sanitize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply the Perfect Label filter.

    Keep rows where EITHER:
      - handover_class == 0  (no HO — always valid, ground truth is stable)
      - handover_success == 1 (HO attempted AND succeeded)
    AND:
      - ping_pong_flag == 0  (no oscillation — label is unambiguous)

    Dropped rows:
      - Failed HOs (success=0, class>0): contradictory physics
      - Ping-pong events (flag=1): ambiguous tie between two cells
    """
    mask = ((df["handover_class"] == 0) | (df["handover_success"] == 1))
    mask = mask & (df["ping_pong_flag"] == 0)
    out  = df[mask].reset_index(drop=True)
    log.info("Sanitization: %d → %d rows (dropped %d noisy labels, %.1f%%)",
             len(df), len(out), len(df)-len(out),
             100*(len(df)-len(out))/len(df))
    return out


def build_global_state_matrix(df: pd.DataFrame) -> np.ndarray:
    """
    Build (N, 9) global state matrix from scalar columns.

    Layout:
      [0] speed       — UE velocity (km/h), signals trajectory intent
      [1] dir_cos     — cos(direction_deg), smooth circular encoding
      [2] dir_sin     — sin(direction_deg), handles 0°/360° wrap-around
      [3] cell_load   — serving cell load, penalises congested cells
      [4] ho_class_0  — no_handover   (one-hot encoding of policy intent)
      [5] ho_class_1  — coverage HO
      [6] ho_class_3  — inter-RAT HO
      [7] ho_class_5  — emergency HO
      [8] ho_class_6  — RLF / link-failure HO

    The one-hot encoding of handover_class is the key "Strategic Logic"
    signal: a coverage-driven HO selects the cell with best RSRP;
    a load-balancing HO may choose a weaker cell with lower load.
    The model must learn to condition its decision on this intent.
    """
    N      = len(df)
    G      = np.zeros((N, HP["G_DIM"]), dtype=np.float32)
    dr     = np.radians(df["direction"].values.astype(np.float32))
    G[:, 0] = df["speed"].values.astype(np.float32)
    G[:, 1] = np.cos(dr)     # dir_cos
    G[:, 2] = np.sin(dr)     # dir_sin
    G[:, 3] = df["cell_load"].values.astype(np.float32)
    for col_idx, cls in enumerate([0, 1, 3, 5, 6]):
        G[:, 4 + col_idx] = (df["handover_class"].values == cls).astype(np.float32)
    return G


def build_cell_feature_matrix(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Build (N, MAX_CELLS, 3) cell features + (N, MAX_CELLS) mask + (N, MAX_CELLS) IDs."""
    log.info("Parsing neighbour columns …")
    NB_R  = parse_list_col(df["nb_rsrps"])
    NB_S  = parse_list_col(df["nb_sinrs"])
    NB_L  = parse_list_col(df["nb_loads"])
    NB_IDS = parse_list_col(df["nb_cell_ids"])
    MASK  = (~np.isnan(NB_R)).astype(np.float32)
    for a in (NB_R, NB_S, NB_L):
        np.nan_to_num(a, nan=0.0, copy=False)
    CELL = np.stack([NB_R, NB_S, NB_L], axis=2)
    return CELL, MASK, NB_IDS


def build_ue_windows(row_idx: np.ndarray,
                     CELL: np.ndarray,
                     GLOB: np.ndarray,
                     MASK: np.ndarray,
                     NB_IDS: np.ndarray,
                     OPT_ID: np.ndarray,
                     OPT_RSRP: np.ndarray,
                     obs: int = HP["OBS_STEPS"],
                     gap: int = 5, tgt: int = 5,
                     mc: int  = HP["MAX_CELLS"],
                     fc: int  = HP["F_CELL"],
                     gd: int  = HP["G_DIM"]):
    n = len(row_idx); n_win = n - obs - gap - tgt + 1
    if n_win <= 0: return None
    
    X = np.zeros((n_win, mc, obs, fc), dtype=np.float32)
    G = np.zeros((n_win, obs, gd), dtype=np.float32)
    M = np.zeros((n_win, mc), dtype=np.float32)
    y = np.full((n_win, tgt), -1, dtype=np.int32)
    r = np.zeros((n_win, tgt), dtype=np.float32)
    
    rng_shuf = np.random.default_rng(42)
    
    valid_mask = np.ones(n_win, dtype=bool)
    
    for w in range(n_win):
        oi = row_idx[w : w + obs]
        ti = row_idx[w + obs + gap : w + obs + gap + tgt]
        
        # Candidates are defined by the LAST step of the observation window
        last_obs_idx = oi[-1]
        c_ids = NB_IDS[last_obs_idx]
        
        p = rng_shuf.permutation(mc)
        c_ids_shuffled = c_ids[p]
        M[w] = MASK[last_obs_idx][p]
        G[w] = GLOB[oi]
        
        # Align features backwards in time for these specific cells
        for t_idx, row_i in enumerate(oi):
            ids_at_t = NB_IDS[row_i]
            for c_idx, cid in enumerate(c_ids_shuffled):
                if np.isnan(cid) or M[w, c_idx] == 0: continue
                matches = np.where(ids_at_t == cid)[0]
                if len(matches) > 0:
                    idx_at_t = matches[0]
                    X[w, c_idx, t_idx] = CELL[row_i, idx_at_t]
                    
        # Find target cells in the shuffled candidate list
        for step in range(tgt):
            target_id = OPT_ID[ti[step]]
            matches = np.where(c_ids_shuffled == target_id)[0]
            if len(matches) > 0:
                y[w, step] = matches[0]
            else:
                valid_mask[w] = False
                break
        r[w] = OPT_RSRP[ti].astype(np.float32)
        
    if not np.any(valid_mask): return None
    
    return X[valid_mask], G[valid_mask], M[valid_mask], y[valid_mask], r[valid_mask]

log.info("Preprocessing helpers defined.")

11:08:42 │ INFO     │ Preprocessing helpers defined.


In [29]:
# ─── Build or load strategic cache ───────────────────────────────────────────

_CACHE   = PATHS["cache"]
_READY   = all((_CACHE / f"{s}.npz").exists() for s in ["train","val","test"])

# Ensure FORCE_REBUILD triggers a rebuild
_rebuild = FORCE_REBUILD or not (_READY and (_CACHE/"meta.json").exists())
if not _rebuild:
    log.info("Strategic cache found → loading.")
else:
    log.info("Building strategic cache from %s", PATHS["raw_csv"])

if _rebuild:
    _t0 = time.time()

    _df = pd.read_csv(str(PATHS["raw_csv"]), low_memory=False)
    _df["timestamp"] = pd.to_datetime(_df["timestamp"], format="mixed")
    _df = _df.sort_values(["ue_id","timestamp"]).reset_index(drop=True)

    # ── SANITIZATION (Perfect Label filter) ──────────────────────────────────
    _df = sanitize_dataframe(_df)

    # ── Build global matrices (vectorised, O(N) not O(N×W)) ──────────────────
    _CELL, _MASK, _NB_IDS = build_cell_feature_matrix(_df)
    _GLOB        = build_global_state_matrix(_df)
    _OPT_ID      = _df["optimal_cell_id"].values
    _OPT_RSRP    = _df["optimal_cell_rsrp"].values.astype(np.float32)
    _UE_GRP      = {uid: grp.index.values
                    for uid, grp in _df.groupby("ue_id")}

    # Load UE split
    _ue = None
    for _sp in [_CACHE/"ue_split.json",
                _ROOT/"dataset"/"processed"/"ue_split.json",
                _ROOT/"dataset"/"deepset_cache"/"ue_split.json"]:
        if _sp.exists():
            with open(str(_sp)) as _f: _ue = json.load(_f)
            break
            
    if _ue is None:
        log.warning("No ue_split.json found. Generating a new split...")
        _ue_list = list(_UE_GRP.keys())
        np.random.default_rng(42).shuffle(_ue_list)
        _n_te, _n_va = int(len(_ue_list)*0.15), int(len(_ue_list)*0.15)
        _te = _ue_list[:_n_te]
        _va = _ue_list[_n_te:_n_te+_n_va]
        _tr = _ue_list[_n_te+_n_va:]
    else:
        _tr, _va, _te = _ue["train_ues"], _ue["val_ues"], _ue["test_ues"]

    def _build_split(ues):
        Xs, Gs, Ms, ys, rs = [], [], [], [], []
        for uid in ues:
            res = build_ue_windows(
                _UE_GRP.get(uid, np.array([])),
                _CELL, _GLOB, _MASK, _NB_IDS, _OPT_ID, _OPT_RSRP)
            if res:
                X,G,M,y,r = res
                Xs.append(X);Gs.append(G);Ms.append(M);ys.append(y);rs.append(r)
        return (np.concatenate(Xs).astype(np.float32),
                np.concatenate(Gs).astype(np.float32),
                np.concatenate(Ms).astype(np.float32),
                np.concatenate(ys).astype(np.int32),
                np.concatenate(rs).astype(np.float32))

    log.info("Building splits …")
    X_tr,G_tr,M_tr,y_tr,r_tr = _build_split(_tr); gc.collect()
    X_va,G_va,M_va,y_va,r_va = _build_split(_va); gc.collect()
    X_te,G_te,M_te,y_te,r_te = _build_split(_te); gc.collect()
    del _df, _CELL, _GLOB, _MASK, _NB_IDS; gc.collect()

    # Scalers — TRAIN only
    _N,_C,_W,_F = X_tr.shape; _G = G_tr.shape[-1]
    _sx = StandardScaler(); _sx.fit(X_tr.reshape(-1,_F))
    _sg = StandardScaler(); _sg.fit(G_tr.reshape(-1,_G))
    _sr = StandardScaler(); _sr.fit(r_tr)

    def _scX(X): n,c,w,f=X.shape; return _sx.transform(X.reshape(-1,f)).reshape(n,c,w,f).astype(np.float32)
    def _scG(G): n,w,g=G.shape;   return _sg.transform(G.reshape(-1,g)).reshape(n,w,g).astype(np.float32)

    X_tr_s=_scX(X_tr);del X_tr; X_va_s=_scX(X_va);del X_va; X_te_s=_scX(X_te);del X_te
    G_tr_s=_scG(G_tr);del G_tr; G_va_s=_scG(G_va);del G_va; G_te_s=_scG(G_te);del G_te
    gc.collect()

    # Sample weights (inverse-frequency, injected into tf.data)
    _cw = compute_class_weight("balanced", classes=np.unique(y_tr.flatten()), y=y_tr.flatten())
    _CW = {int(c):float(w) for c,w in enumerate(_cw)}
    def _sw(y): return np.mean([[_CW.get(int(yi),1.) for yi in row] for row in y], axis=1).astype(np.float32)

    for s,X,G,M,y,r,sw in [
        ("train",X_tr_s,G_tr_s,M_tr,y_tr,_sr.transform(r_tr).astype(np.float32),_sw(y_tr)),
        ("val",  X_va_s,G_va_s,M_va,y_va,_sr.transform(r_va).astype(np.float32),_sw(y_va)),
        ("test", X_te_s,G_te_s,M_te,y_te,_sr.transform(r_te).astype(np.float32),_sw(y_te)),
    ]:
        np.savez_compressed(str(_CACHE/f"{s}.npz"), X=X,G=G,M=M,y=y,r=r,sw=sw)

    with open(str(_CACHE/"scalers.pkl"),"wb") as _f:
        pickle.dump({"X":_sx,"G":_sg,"r":_sr,"cw_map":_CW},_f)
    json.dump({"F_CELL":HP["F_CELL"],"G_DIM":HP["G_DIM"],"OBS_STEPS":HP["OBS_STEPS"],
               "cw_map":_CW,"sanitization":{"handover_success":1,"ping_pong_flag":0}},
              open(str(_CACHE/"meta.json"),"w"),indent=2)
    json.dump({"train_ues":_tr,"val_ues":_va,"test_ues":_te},
              open(str(_CACHE/"ue_split.json"),"w"),indent=2)
    log.info("Cache built in %.1f s", time.time()-_t0)

# ── Load ──────────────────────────────────────────────────────────────────────
log.info("Loading strategic cache …")
_tr=np.load(str(_CACHE/"train.npz")); _va=np.load(str(_CACHE/"val.npz")); _te=np.load(str(_CACHE/"test.npz"))

X_tr,G_tr,M_tr,y_tr,r_tr,sw_tr = _tr["X"],_tr["G"],_tr["M"],_tr["y"].astype(np.int32),_tr["r"],_tr["sw"]
X_va,G_va,M_va,y_va,r_va,sw_va = _va["X"],_va["G"],_va["M"],_va["y"].astype(np.int32),_va["r"],_va["sw"]
X_te,G_te,M_te,y_te,r_te,sw_te = _te["X"],_te["G"],_te["M"],_te["y"].astype(np.int32),_te["r"],_te["sw"]

with open(str(_CACHE/"scalers.pkl"),"rb") as _f: _sc=pickle.load(_f)
scaler_X=_sc["X"]; scaler_G=_sc["G"]; scaler_r=_sc["r"]; CW_MAP=_sc["cw_map"]

for t,(X,G,M,y) in [("train",(X_tr,G_tr,M_tr,y_tr)),
                     ("val",  (X_va,G_va,M_va,y_va)),
                     ("test", (X_te,G_te,M_te,y_te))]:
    log.info("  %-5s: X=%s  G=%s  M=%s  y=%s", t, X.shape, G.shape, M.shape, y.shape)

11:08:42 │ INFO     │ Building strategic cache from /home/wassimmchichi/Downloads/Handover_projects/dataset/raw/handover_dataset.csv
11:08:44 │ INFO     │ Sanitization: 90300 → 85518 rows (dropped 4782 noisy labels, 5.3%)
11:08:44 │ INFO     │ Parsing neighbour columns …
11:08:46 │ INFO     │ Building splits …
11:10:43 │ INFO     │ Cache built in 121.0 s
11:10:43 │ INFO     │ Loading strategic cache …
11:10:44 │ INFO     │   train: X=(43597, 10, 25, 3)  G=(43597, 25, 9)  M=(43597, 10)  y=(43597, 5)
11:10:44 │ INFO     │   val  : X=(8992, 10, 25, 3)  G=(8992, 25, 9)  M=(8992, 10)  y=(8992, 5)
11:10:44 │ INFO     │   test : X=(9385, 10, 25, 3)  G=(9385, 25, 9)  M=(9385, 10)  y=(9385, 5)


## Section 4 · tf.data Pipeline — Strategic 3-Tuple with Sample Weights

In [30]:
# ─── Section 4 · tf.data pipeline ────────────────────────────────────────────
#
# Dataset returns (inputs, targets, sample_weights) — 3-tuple.
# This bypasses the Keras class_weight ValueError on multi-output models.
#
# Inputs  : {"cells": X, "global_state": G, "mask": M}
# Targets : {"cls_output": y_one_hot, "reg_rsrp": r}
# Weights : {"cls_output": sw}  — per-sample inverse-frequency weight

AUTOTUNE = tf.data.AUTOTUNE


def make_ds(X: np.ndarray, G: np.ndarray, M: np.ndarray,
            y: np.ndarray, r: np.ndarray, sw: np.ndarray,
            shuffle: bool = False) -> tf.data.Dataset:
    y_oh = tf.one_hot(y, depth=HP["MAX_CELLS"]).numpy().astype(np.float32)
    ds   = tf.data.Dataset.from_tensor_slices((
        {"cells": X, "global_state": G, "mask": M},
        {"cls_output": y_oh, "reg_rsrp": r.astype(np.float32)},
        {"cls_output": sw.astype(np.float32)},
    ))
    if shuffle:
        ds = ds.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(HP["BATCH_SIZE"], drop_remainder=False).prefetch(AUTOTUNE)


ds_tr = make_ds(X_tr, G_tr, M_tr, y_tr, r_tr, sw_tr, shuffle=True)
ds_va = make_ds(X_va, G_va, M_va, y_va, r_va, sw_va)
ds_te = make_ds(X_te, G_te, M_te, y_te, r_te, sw_te)

steps_per_epoch = len(ds_tr)
log.info("Batches → train:%d  val:%d  test:%d", len(ds_tr),len(ds_va),len(ds_te))
log.info("Class weights: %s", {int(k):round(float(v),3) for k,v in CW_MAP.items()})

# Label distribution chart
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, (y_sp, title) in zip(axes, [(y_tr,"Train (sanitized)"),(y_va,"Val")]):
    vals, cnts = np.unique(y_sp, return_counts=True)
    ax.bar(vals, cnts, color=["#E24B4A" if v==0 else "#378ADD" for v in vals], edgecolor="white")
    ax.set_xticks(vals); ax.set_xticklabels([f"C{v}" for v in vals], fontsize=9)
    ax.set_title(title, fontweight="bold"); ax.set_xlabel("Cell index"); ax.set_ylabel("Count")
plt.suptitle("Label distribution after Perfect Label sanitization", fontweight="bold")
plt.tight_layout()
plt.savefig(str(PATHS["metrics"]/"strategic_label_dist.png"),dpi=150,bbox_inches="tight")
plt.close()

11:10:44 │ INFO     │ Batches → train:341  val:71  test:74
11:10:44 │ INFO     │ Class weights: {0: 0.861, 1: 1.159, 2: 0.911, 3: 1.135, 4: 1.03, 5: 1.065, 6: 1.223, 7: 0.897, 8: 0.854, 9: 1.017}


## Section 5 · Loss Functions

### Focal Loss — fixing Cell 0 dominance (Cell 0 recall = 83%, Cells 2-7 < 30%)

```
FL(p_t) = −α·(1−p_t)^γ·log(p_t)

With γ=2:
  Cell 0 (p_t ≈ 0.95) → weight = 0.25·(0.05)² = 0.000625  (down-weighted ×160×)
  Cell 7 (p_t ≈ 0.10) → weight = 0.25·(0.90)² = 0.2025    (near full weight)
```

Focal Loss alone is insufficient because it operates on mini-batch statistics.
The sample weights (pre-computed inverse-frequency) in `tf.data` add a second
layer of rebalancing at the dataset level.

### Huber Loss — RSRP regression auxiliary head

```
L_Huber(e) = { ½e²           if |e| ≤ δ = 1.0
             { δ·(|e| − ½δ)  otherwise

```
Drone cells produce extreme RSRP values (−125 dBm). MSE would amplify these
outlier gradients and dominate the joint loss. Huber behaves like L2 for small
errors and like L1 for large ones — the auxiliary head provides physics grounding
without being dominated by outliers.

In [31]:
# ─── Section 5 · Loss functions ──────────────────────────────────────────────

def focal_loss(gamma: float = 2.0, alpha: float = 0.25):
    """Multi-class Focal Loss for one-hot targets."""
    def _loss(y_true, y_pred):
        y_pred  = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1-1e-7)
        y_true  = tf.cast(y_true, tf.float32)
        ce      = -y_true * tf.math.log(y_pred)
        p_t     = tf.reduce_sum(y_true*y_pred, axis=-1, keepdims=True)
        weight  = alpha * tf.pow(1.0-p_t, gamma)
        return tf.reduce_sum(weight*ce, axis=-1)
    _loss.__name__ = f"focal_g{gamma}_a{alpha}"
    return _loss


FOCAL_LOSS = focal_loss(HP["FOCAL_GAMMA"], HP["FOCAL_ALPHA"])
HUBER_LOSS = keras.losses.Huber(delta=HP["HUBER_DELTA"])

log.info("Focal Loss: γ=%.1f  α=%.2f", HP["FOCAL_GAMMA"], HP["FOCAL_ALPHA"])
log.info("Huber Loss: δ=%.1f", HP["HUBER_DELTA"])

11:10:45 │ INFO     │ Focal Loss: γ=2.0  α=0.25
11:10:45 │ INFO     │ Huber Loss: δ=1.0


## Section 6 · Strategic DeepSet Architecture

### The key architectural change: Context Injection

```
BEFORE (vanilla DeepSet):                AFTER (Strategic DeepSet):
─────────────────────────────────        ──────────────────────────────────────────────
z = GlobalAvgPool(Φ(h_i))               z_glob = LSTM(global_state)  ← NEW
                                          z_pool = MaskedAvgPool(Φ(h_i))
                                          z_aug  = Concat([z_pool, z_glob])  ← AUGMENTED

ρ input: [Φ(h_i) ‖ z]     (128-dim)     ρ input: [Φ(h_i) ‖ z_aug]   (64+64+32 = 160-dim)
ρ Dense: 64                              ρ Dense: 256 → 128  ← ENLARGED capacity
```

### Why inject `global_state` as a time-series through LSTM (not as a scalar)?

The global state (speed, direction, handover_class) evolves over the 25-step
observation window. A UE may have accelerated in the last 5 steps, or the
handover_class may have changed mid-window as the simulator's A3 event fires.

A single scalar summary would miss this temporal evolution. Running it through
a dedicated LSTM(32) captures the _trajectory_ of the policy decision, not
just its current value.

In [32]:
# ─── Section 6 · Custom layers ───────────────────────────────────────────────

class MaskedGlobalAveragePooling(keras.layers.Layer):
    """
    Correct DeepSet aggregation: z = Σ Φ(hᵢ)·maskᵢ / Σ maskᵢ

    Divides by the actual count of real cells, not MAX_CELLS=10.
    This prevents 2-cell windows from being diluted by 8 zero-padded slots.
    """
    def call(self, phi, mask):
        summed = tf.reduce_sum(phi * mask, axis=1)
        count  = tf.maximum(tf.reduce_sum(mask, axis=1), 1e-8)
        return summed / count

    def get_config(self): return super().get_config()


log.info("MaskedGlobalAveragePooling defined.")

11:10:45 │ INFO     │ MaskedGlobalAveragePooling defined.


In [33]:
# ─── Section 6b · Strategic DeepSet model ────────────────────────────────────

def build_strategic_deepset(hp: dict) -> keras.Model:
    """
    Strategic DeepSet with Context Injection.

    Architecture stages:

    Stage 1 — Temporal Encoder (shared, per-cell):
        TimeDistributed(LSTM(64)) — captures 5s RF trend per candidate cell.
        Same weights for all cells → permutation invariant.

    Stage 2 — Φ projection (shared, per-cell):
        TimeDistributed(Dense(64)) — non-linear embedding.

    Stage 3 — Global State Encoder (separate LSTM):
        LSTM(32) on the global_state time-series (speed, direction, ho_class, load).
        This encodes the UE's TRAJECTORY INTENT and simulator POLICY DECISION.
        Its output is concatenated into the global context vector z.

    Stage 4 — Masked Global Average Pool:
        z_pool = Σ Φ(hᵢ)·maskᵢ / Σ maskᵢ
        Permutation-invariant summary of the RF environment.

    Stage 5 — Context Injection:
        z_aug = Concat([z_pool, z_glob])  → 64 + 32 = 96 dims
        Every cell's scoring now conditions on both RF environment AND policy.

    Stage 6 — ρ Decoder (enlarged):
        [Φ(hᵢ) ‖ z_aug] → Dense(256) → Dense(128) → Dense(1 logit)
        Enlarged from Dense(64) to handle the 160-dim concatenated input.

    Stage 7 — Masked Softmax:
        Padding cells receive −1e9 before softmax → zero probability.

    Auxiliary Head — RSRP Regression:
        Pool → Dense(32) → Dense(1) — predicts target-cell RSRP.
        Forces encoder to understand signal magnitude, not just ranking.
    """
    C  = hp["MAX_CELLS"]
    W  = hp["OBS_STEPS"]
    FC = hp["F_CELL"]
    GD = hp["G_DIM"]
    D  = hp["PHI_DIM"]    # 64

    # ── Inputs ────────────────────────────────────────────────────────────────
    inp_cells = keras.Input((C, W, FC), name="cells",        dtype="float32")
    inp_glob  = keras.Input((W, GD),    name="global_state", dtype="float32")
    inp_mask  = keras.Input((C,),       name="mask",         dtype="float32")

    # ── Stage 1: Temporal Encoder — shared LSTM across cells ─────────────────
    trend = layers.TimeDistributed(
        layers.LSTM(hp["LSTM_CELL"], return_sequences=False),
        name="td_lstm")(inp_cells)                              # (B, C, 64)

    # ── Stage 2: Φ — shared per-cell MLP ─────────────────────────────────────
    phi = layers.TimeDistributed(
        layers.Dense(D, activation="relu"), name="phi")(trend)
    phi = layers.TimeDistributed(
        layers.Dropout(hp["DROPOUT"]), name="phi_drop")(phi)    # (B, C, 64)

    # ── Stage 3: Global State Encoder — trajectory + policy LSTM ─────────────
    z_glob = layers.LSTM(hp["LSTM_GLOB"], return_sequences=False,
                          name="glob_lstm")(inp_glob)            # (B, 32)
    z_glob = layers.Dense(hp["LSTM_GLOB"], activation="relu",
                           name="glob_proj")(z_glob)             # (B, 32)

    # ── Stage 4: Masked Global Average Pool ──────────────────────────────────
    mask_exp = layers.Reshape((C, 1), name="mask_exp")(inp_mask)
    pool_fn  = MaskedGlobalAveragePooling(name="masked_pool")
    z_pool   = pool_fn(phi, mask_exp)                            # (B, 64)

    # ── Stage 5: Context Injection ────────────────────────────────────────────
    z_aug    = layers.Concatenate(name="z_aug")([z_pool, z_glob])  # (B, 96)
    z_aug    = layers.Dense(96, activation="relu", name="z_dense")(z_aug)
    z_tiled  = layers.RepeatVector(C, name="z_tile")(z_aug)        # (B, C, 96)

    # ── Stage 6: ρ Decoder — enlarged to handle 160-dim input ────────────────
    rho = layers.Concatenate(axis=-1, name="rho_concat")([phi, z_tiled])
    # rho: (B, C, 64 + 96) = (B, C, 160)
    for i, units in enumerate(hp["RHO_DIMS"]):
        rho = layers.TimeDistributed(
            layers.Dense(units, activation="relu"), name=f"rho_{i}")(rho)
        rho = layers.TimeDistributed(
            layers.Dropout(hp["DROPOUT"]), name=f"rho_drop_{i}")(rho)
    H = hp.get("TGT_STEPS", 5)
    logits = layers.TimeDistributed(
        layers.Dense(H, use_bias=True), name="scorer")(rho)
    logits = layers.Permute((2, 1), name="logits_permuted")(logits)  # (B, H, C)

    # ── Stage 7: Masked Softmax ───────────────────────────────────────────────
    cls_out = layers.Softmax(axis=-1, dtype="float32", name="cls_output")(
        layers.Add(name="pad_mask")([logits, layers.Reshape((1, C))((1.0 - inp_mask) * (-1e9))]))

    # ── Auxiliary Head: RSRP Regression ──────────────────────────────────────
    z_reg    = layers.Dense(32, activation="relu",  name="reg_fc")(z_aug)
    z_reg    = layers.Dropout(hp["DROPOUT"],        name="reg_drop")(z_reg)
    H = hp.get("TGT_STEPS", 5)
    reg_out  = layers.Dense(H, activation=None,
                              dtype="float32", name="reg_rsrp")(z_reg)

    model = keras.Model(
        inputs  = [inp_cells, inp_glob, inp_mask],
        outputs = {"cls_output": cls_out, "reg_rsrp": reg_out},
        name    = "Strategic_DeepSet_HO",
    )
    return model


model = build_strategic_deepset(HP)
model.summary(line_length=90, expand_nested=False)
log.info("Parameters: %d", model.count_params())
log.info("Cell encoder: LSTM(%d) + Dense(%d)", HP["LSTM_CELL"], HP["PHI_DIM"])
log.info("Global encoder: LSTM(%d)", HP["LSTM_GLOB"])
log.info("ρ head: %s", HP["RHO_DIMS"])

Model: "Strategic_DeepSet_HO"
__________________________________________________________________________________________
 Layer (type)              Output Shape               Param   Connected to                
                                                       #                                  
 cells (InputLayer)        [(None, 10, 25, 3)]        0       []                          
                                                                                          
 td_lstm (TimeDistributed  (None, 10, 64)             17408   ['cells[0][0]']             
 )                                                                                        
                                                                                          
 phi (TimeDistributed)     (None, 10, 64)             4160    ['td_lstm[0][0]']           
                                                                                          
 mask (InputLayer)         [(None, 10)]               0     

## Section 7 · Compile — Weighted Loss + Warm-Up Schedule

### Total loss formula
```
L_total = 1.0 × L_focal(cls_output)  +  0.5 × L_huber(reg_rsrp)
```

### Two-layer imbalance correction
1. **Focal Loss (γ=2)** — down-weights confident Cell 0 predictions per-batch
2. **Sample Weights (tf.data 3-tuple)** — inverse-frequency reweighting at dataset level

Both layers work independently:
- Focal operates on gradient magnitude per sample
- Sample weights scale the loss contribution of each sample before backprop

This dual correction is necessary because the imbalance ratio is ~50:1 (Cell 0 vs Cell 7).

In [34]:
# ─── Section 7 · LR schedule + compile ───────────────────────────────────────

class WarmUpCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self,lr_max,lr_min,warmup_steps,decay_steps):
        super().__init__()
        self.lr_max=float(lr_max);self.lr_min=float(lr_min)
        self.warmup_steps=float(warmup_steps);self.decay_steps=float(decay_steps)
    def __call__(self,step):
        step=tf.cast(step,tf.float32)
        warmup=self.lr_max*step/tf.maximum(self.warmup_steps,1.0)
        cosine=self.lr_min+0.5*(self.lr_max-self.lr_min)*(
            1.0+tf.cos(np.pi*tf.minimum(step-self.warmup_steps,
                                         self.decay_steps)/self.decay_steps))
        return tf.where(step<self.warmup_steps,warmup,cosine)
    def get_config(self):
        return {"lr_max":self.lr_max,"lr_min":self.lr_min,
                "warmup_steps":self.warmup_steps,"decay_steps":self.decay_steps}


ws = HP["LR_WARMUP_EP"] * steps_per_epoch
ds_ = HP["LR_DECAY_EP"]  * steps_per_epoch
lr_sched = WarmUpCosineDecay(HP["LR_INIT"], HP["LR_INIT"]*0.01, ws, ds_)

model.compile(
    optimizer    = keras.optimizers.Adam(learning_rate=lr_sched),
    loss         = {"cls_output": FOCAL_LOSS, "reg_rsrp": HUBER_LOSS},
    loss_weights = {"cls_output": HP["LAMBDA_CLS"], "reg_rsrp": HP["LAMBDA_REG"]},
    metrics      = {
        "cls_output": [
            keras.metrics.CategoricalAccuracy(name="top1_acc"),
            keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
        ],
        "reg_rsrp": [keras.metrics.MeanAbsoluteError(name="rsrp_mae")],
    },
)
log.info("Compiled. L=%.1f×Focal + %.1f×Huber  LR=%.0e  warmup=%dep",
         HP["LAMBDA_CLS"], HP["LAMBDA_REG"], HP["LR_INIT"], HP["LR_WARMUP_EP"])

11:10:46 │ INFO     │ Compiled. L=1.0×Focal + 0.5×Huber  LR=5e-04  warmup=4ep


## Section 9 · Training

In [35]:
# ─── Section 9 · Callbacks & training ────────────────────────────────────────

MONITOR = "val_cls_output_top1_acc"

class MLflowEpochCB(keras.callbacks.Callback):
    def on_epoch_end(self,epoch,logs=None):
        if MLFLOW_OK and logs:
            for k,v in logs.items(): mlflow.log_metric(k,float(v),step=epoch)

if MLFLOW_OK:
    _run=mlflow.start_run(run_name="strategic_deepset_exp6")
    mlflow.log_params(HP)
    mlflow.log_param("sanitization","success+no_ho+no_pingpong")
    mlflow.log_param("global_state","speed+dir_cos+dir_sin+cell_load+ho_class5")
    mlflow.log_param("imbalance","focal_g2+sample_weights_3tuple")
    _rid=_run.info.run_id; log.info("MLflow: %s",_rid)
else: _rid=None

callbacks=[
    keras.callbacks.EarlyStopping(
        monitor=MONITOR,patience=12,min_delta=1e-4,
        restore_best_weights=True,mode="max",verbose=1),
    keras.callbacks.ModelCheckpoint(
        filepath=CKPT_PATH,monitor=MONITOR,
        save_best_only=True,mode="max",verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor=MONITOR,factor=0.5,patience=6,min_lr=1e-7,mode="max",verbose=1),
    keras.callbacks.TensorBoard(
        log_dir=str(PATHS["tb_logs"]),histogram_freq=0,
        write_graph=True,update_freq="epoch"),
    keras.callbacks.CSVLogger(
        str(PATHS["metrics"]/"training_log.csv"),append=False),
    MLflowEpochCB(),
]
log.info("Monitor='%s'  CKPT='%s'", MONITOR, CKPT_PATH)

# class_weight is NOT passed to model.fit() — it would raise
# ValueError on multi-output models. Rebalancing is handled via
# Focal Loss + sample_weights inside the tf.data 3-tuple Dataset.
history=model.fit(
    ds_tr,
    validation_data=ds_va,
    epochs=HP["EPOCHS"],
    callbacks=callbacks,
    verbose=1,
)

best_ep=int(np.argmax(history.history[MONITOR]))+1
best_top1=float(max(history.history[MONITOR]))
log.info("Done — best %s=%.4f @ ep %d", MONITOR, best_top1, best_ep)
if MLFLOW_OK:
    mlflow.log_metrics({"best_val_top1.keras":best_top1,"best_epoch.keras":float(best_ep)})

11:10:46 │ INFO     │ MLflow: 35b36f6ff15c4fb7865d9c779242c7a5
11:10:46 │ INFO     │ Monitor='val_cls_output_top1_acc'  CKPT='/home/wassimmchichi/Downloads/Handover_projects/models/best_strategic_deepset.keras'
Epoch 1/60
  5/341 [..............................] - ETA: 4s - loss: 0.5402 - cls_output_loss: 0.3261 - reg_rsrp_loss: 0.4280 - cls_output_top1_acc: 0.2559 - cls_output_top3_acc: 0.5419 - reg_rsrp_rsrp_mae: 0.8128   WARNING:tensorflow:Callback method `on_train_batch_end` is slow compared to the batch time (batch time: 0.0117s vs `on_train_batch_end` time: 0.0185s). Check your callbacks.
11:10:54 │ WARNING  │ Callback method `on_train_batch_end` is slow compared to the batch time (batch time: 0.0117s vs `on_train_batch_end` time: 0.0185s). Check your callbacks.
338/341 [============================>.] - ETA: 0s - loss: 0.4607 - cls_output_loss: 0.2755 - reg_rsrp_loss: 0.3704 - cls_output_top1_acc: 0.3989 - cls_output_top3_acc: 0.7110 - reg_rsrp_rsrp_mae: 0.7378WARNING:tensorflow

In [36]:
_out = PATHS["metrics"] / "strategic-deepset-architecture.png"

tf.keras.utils.plot_model(
    model,
    to_file=str(_out),
    show_shapes=True,
    show_layer_names=True,
    dpi=150
)

log.info("Saved: %s", _out)

if MLFLOW_OK:
    mlflow.log_artifact(str(_out))

11:13:26 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/strategic_deepset/strategic-deepset-architecture.png


## Section 10 · Evaluation — Classification Report & Per-Cell Recall

In [37]:
# ─── Section 10 · Evaluation ─────────────────────────────────────────────────

model = keras.models.load_model(
    CKPT_PATH,
    custom_objects={
        "MaskedGlobalAveragePooling": MaskedGlobalAveragePooling,
        "WarmUpCosineDecay": WarmUpCosineDecay,
        # Add the focal loss function instance
        FOCAL_LOSS.__name__: FOCAL_LOSS 
    },
)
log.info("Loaded: %s", CKPT_PATH)


preds=model.predict(ds_te,verbose=1)
probs_te=preds["cls_output"]; rsrp_pred=preds["reg_rsrp"]
y_pred_te=probs_te.argmax(axis=-1)

y_te_flat = y_te.ravel()
y_pred_te_flat = y_pred_te.ravel()
probs_te_flat = probs_te.reshape(-1, probs_te.shape[-1])

top1=float((y_pred_te_flat==y_te_flat).mean())
top3=float(top_k_accuracy_score(y_te_flat,probs_te_flat,k=3,labels=ALL_LABELS))
top5=float(top_k_accuracy_score(y_te_flat,probs_te_flat,k=5,labels=ALL_LABELS))

rsrp_dbm_pred = scaler_r.inverse_transform(rsrp_pred)
rsrp_dbm_true = scaler_r.inverse_transform(r_te)

rsrp_mae = float(
    mean_absolute_error(
        rsrp_dbm_true.ravel(),
        rsrp_dbm_pred.ravel()
    )
)
rsrp_mae=float(mean_absolute_error(rsrp_dbm_true.ravel(),rsrp_dbm_pred.ravel()))

log.info("Test → Top-1:%.4f  Top-3:%.4f  RSRP_MAE:%.2f dBm", top1,top3,rsrp_mae)
if MLFLOW_OK:
    mlflow.log_metrics({"test_top1":top1,"test_top3":top3,"test_top5":top5,
                         "test_rsrp_mae_dbm":rsrp_mae})

print("="*65)
print("  STRATEGIC DEEPSET — TEST RESULTS")
print("="*65)
print(f"  Top-1 : {top1:.4f}  ({top1*100:.2f}%)   previous: 55%")
print(f"  Top-3 : {top3:.4f}  ({top3*100:.2f}%)")
print(f"  Top-5 : {top5:.4f}  ({top5*100:.2f}%)")
print(f"  RSRP MAE: {rsrp_mae:.2f} dBm")
print()
print(classification_report(
    y_te_flat, 
    y_pred_te_flat,
    # Explicitly list all possible labels (0-9)
    labels=list(range(HP["MAX_CELLS"])), 
    target_names=[f"Cell{i}" for i in range(HP["MAX_CELLS"])],
    digits=4, 
    zero_division=0
))


11:13:30 │ INFO     │ Loaded: /home/wassimmchichi/Downloads/Handover_projects/models/best_strategic_deepset.keras
74/74 [==============================] - 1s 4ms/step
11:13:31 │ INFO     │ Test → Top-1:0.5135  Top-3:0.8668  RSRP_MAE:4.94 dBm
  STRATEGIC DEEPSET — TEST RESULTS
  Top-1 : 0.5135  (51.35%)   previous: 55%
  Top-3 : 0.8668  (86.68%)
  Top-5 : 0.9559  (95.59%)
  RSRP MAE: 4.94 dBm

              precision    recall  f1-score   support

       Cell0     0.5443    0.5846    0.5637      5279
       Cell1     0.4786    0.4462    0.4619      4036
       Cell2     0.5241    0.5555    0.5393      5215
       Cell3     0.4787    0.4772    0.4780      4078
       Cell4     0.5173    0.5003    0.5087      4599
       Cell5     0.5196    0.5131    0.5163      4520
       Cell6     0.4795    0.4495    0.4640      3900
       Cell7     0.5232    0.5391    0.5310      5144
       Cell8     0.5352    0.5386    0.5369      5451
       Cell9     0.5029    0.4861    0.4943      4703

    accu

In [38]:
print("rsrp_pred:", rsrp_pred.shape)
print("r_te:", r_te.shape)
print("scaler mean:", scaler_r.mean_.shape)
print("scaler scale:", scaler_r.scale_.shape)

rsrp_pred: (9385, 5)
r_te: (9385, 5)
scaler mean: (5,)
scaler scale: (5,)


## Section 11 · Per-Cell Recall Analysis & Comparison

In [39]:
# ─── Section 11 · Per-cell recall & experiment comparison ────────────────────

C=HP["MAX_CELLS"]
cm=confusion_matrix(y_te_flat,y_pred_te_flat,labels=list(range(C)))
cm_norm=cm.astype(float)/(cm.sum(axis=1,keepdims=True)+1e-9)
cl=[f"C{i}" for i in range(C)]

fig,axes=plt.subplots(1,2,figsize=(16,6))
import seaborn as sns
sns.heatmap(cm_norm,annot=True,fmt=".2f",cmap="YlGn",
            xticklabels=[f"P-{l}" for l in cl],
            yticklabels=[f"T-{l}" for l in cl],
            linewidths=0.5,ax=axes[0],vmin=0,vmax=1,annot_kws={"size":9})
axes[0].set(title="Normalised Confusion — Strategic DeepSet",
            ylabel="Actual",xlabel="Predicted")

per_recall=cm_norm.diagonal()
BASELINE_RECALL = [
    0.83, 0.30, 0.15, 0.10, 0.05,
    0.05, 0.03, 0.02, 0.01, 0.01
]
x=np.arange(C); w=0.35
axes[1].bar(x-w/2,BASELINE_RECALL,w,label="Baseline DeepSet",
            color="#B0BEC5",edgecolor="white")
axes[1].bar(x+w/2,per_recall,w,label="Strategic DeepSet",
            color=["#2196F3" if v>=0.5 else "#DD8452" for v in per_recall],
            edgecolor="white")
axes[1].axhline(top1,color="black",ls="--",lw=1.2,
                label=f"Overall Top-1 ({top1:.3f})")
for i,(b,s) in enumerate(zip(BASELINE_RECALL,per_recall)):
    axes[1].text(i+w/2,s+0.015,f"{s:.2f}",ha="center",fontsize=7,fontweight="bold")
axes[1].set(xticks=x,xticklabels=[f"C{i}" for i in range(C)],
            ylabel="Recall",ylim=(0,1.15),
            title="Per-Cell Recall: Baseline vs Strategic DeepSet")
axes[1].legend(fontsize=8); axes[1].grid(axis="y",alpha=0.4)
plt.xticks(rotation=0); plt.tight_layout()
_out=PATHS["metrics"]/"strategic_per_cell_recall.png"
plt.savefig(str(_out),dpi=150,bbox_inches="tight"); plt.close()
if MLFLOW_OK: mlflow.log_artifact(str(_out))

print("Per-cell recall comparison:")
print(f"{'Cell':>6} {'Baseline':>10} {'Strategic':>10} {'Delta':>8}")
for i,(b,s) in enumerate(zip(BASELINE_RECALL,per_recall)):
    delta=s-b
    marker="↑" if delta>0.05 else ("↓" if delta<-0.05 else "→")
    print(f"  C{i}    {b:>8.3f}    {s:>8.3f}    {delta:>+7.3f} {marker}")

Per-cell recall comparison:
  Cell   Baseline  Strategic    Delta
  C0       0.830       0.585     -0.245 ↓
  C1       0.300       0.446     +0.146 ↑
  C2       0.150       0.556     +0.406 ↑
  C3       0.100       0.477     +0.377 ↑
  C4       0.050       0.500     +0.450 ↑
  C5       0.050       0.513     +0.463 ↑
  C6       0.030       0.449     +0.419 ↑
  C7       0.020       0.539     +0.519 ↑
  C8       0.010       0.539     +0.529 ↑
  C9       0.010       0.486     +0.476 ↑


## Section 12 · Save & MLflow Close

In [40]:
# ─── Section 12 · Save ───────────────────────────────────────────────────────

model.save(FINAL_PATH)

# Training curves
hist=history.history; ep=range(1,len(hist["loss"])+1)
fig,axes=plt.subplots(1,3,figsize=(16,4.5))
for ax,(tr_k,va_k,title,hi) in zip(axes,[
    ("loss","val_loss","Total Loss",False),
    ("cls_output_top1_acc","val_cls_output_top1_acc","Top-1 Accuracy",True),
    ("cls_output_top3_acc","val_cls_output_top3_acc","Top-3 Accuracy",True),
]):
    if tr_k not in hist: continue
    ax.plot(ep,hist[tr_k],lw=2,label="train")
    ax.plot(ep,hist[va_k],lw=2,ls="--",label="val")
    fn=np.argmax if hi else np.argmin
    be=fn(hist[va_k])+1; bv=(max if hi else min)(hist[va_k])
    ax.axvline(be,color="red",ls=":",lw=1.2)
    ax.scatter([be],[bv],color="red",zorder=5,s=60,label=f"best({bv:.4f})")
    ax.set(title=title,xlabel="Epoch"); ax.legend(fontsize=8); ax.grid(alpha=0.4)
fig.suptitle("Strategic DeepSet — Training History",fontsize=13,fontweight="bold")
plt.tight_layout()
_out=PATHS["metrics"]/"strategic_training_curves.png"
plt.savefig(str(_out),dpi=150,bbox_inches="tight"); plt.close()

meta={
    "experiment"     : "Exp 6 — Strategic DeepSet (Context Injection)",
    "created"        : datetime.datetime.now().isoformat(),
    "notebook"       : "notebooks/modeling/06_strategic_deepset.ipynb",
    "sanitization"   : "success=1 OR class=0, ping_pong=0",
    "global_state"   : "speed+dir_cos+dir_sin+cell_load+ho_class_onehot",
    "loss"           : f"{HP['LAMBDA_CLS']}xFocal(g{HP['FOCAL_GAMMA']})+{HP['LAMBDA_REG']}xHuber",
    "imbalance"      : "Focal + sample_weights_3tuple",
    "test_top1"      : round(top1,4), "test_top3":round(top3,4),
    "test_top5"      : round(top5,4), "test_rsrp_mae_dbm":round(rsrp_mae,3),
    "hyperparams"    : HP,
    "paths"          : {k:str(v) for k,v in PATHS.items()},
}
_mout=PATHS["metrics"]/"strategic_metadata.json"
json.dump(meta,open(str(_mout),"w"),indent=2)

if MLFLOW_OK:
    for p in PATHS["metrics"].glob("*.png"): mlflow.log_artifact(str(p))
    mlflow.log_artifact(str(_mout))
    mlflow.tensorflow.log_model(model,artifact_path="strategic_deepset",
                                registered_model_name="handover_strategic_deepset")
    mlflow.end_run()
    log.info("MLflow run closed.")

print()
print("="*65)
print(f"  Top-1 : {top1:.4f}   (target: >55%   delta: {top1-0.55:+.4f})")
print(f"  Top-3 : {top3:.4f}")
print(f"  RSRP MAE: {rsrp_mae:.2f} dBm")
print(f"  Overfit diagnostic: {diag_best:.4f}")
print()
print("  TensorBoard: tensorboard --logdir ../../tb_logs/")
print("  MLflow:      mlflow ui --backend-store-uri file://$(pwd)/../../mlflow/mlruns")

2026/05/30 11:13:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 11:13:43 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: /tmp/tmpquq_f3fn/model/data/model/assets
11:13:51 │ INFO     │ Assets written to: /tmp/tmpquq_f3fn/model/data/model/assets


2026/05/30 11:14:01 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpquq_f3fn/model, flavor: tensorflow). Fall back to return ['tensorflow==2.15.1', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
Registered model 'handover_strategic_deepset' already exists. Creating a new version of this model...
2026/05/30 11:14:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: handover_strategic_deepset, version 2
Created version '2' of model 'handover_strategic_deepset'.


🏃 View run strategic_deepset_exp6 at: http://127.0.0.1:5000/#/experiments/16/runs/35b36f6ff15c4fb7865d9c779242c7a5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/16
11:14:28 │ INFO     │ MLflow run closed.

  Top-1 : 0.5135   (target: >55%   delta: -0.0365)
  Top-3 : 0.8668
  RSRP MAE: 4.94 dBm
  Overfit diagnostic: 0.6250

  TensorBoard: tensorboard --logdir ../../tb_logs/
  MLflow:      mlflow ui --backend-store-uri file://$(pwd)/../../mlflow/mlruns
